# C0-v1 development cross-validation: read-only diagnostics

## Objective
Inspect the completed four-arm, five-fold, three-seed experiment. All 60 pipelines completed. This notebook loads only small aggregate artifacts, never patient-level inputs, checkpoints or training code. It performs no fitting, model selection or configuration changes.

NQE Q−E: validation ROC-AUC +0.01510, log loss −0.01050 nats. Hybrid Q−E: ROC-AUC −0.00909, log loss +0.02095 nats. Five-fold seed-averaged variation is descriptive; the folds and seeds are not independent replications. The fixed NQE-Q pipeline does not improve both co-primary endpoints over historical B. No quantum advantage is established.

## Load existing artifacts
Set the configurable relative summary directory below. Missing inputs produce an informative error; no automatic regeneration or training is attempted.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise FileNotFoundError('Run this notebook from the repository root or notebooks/ directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
SUMMARY_DIR = ROOT / 'reports/phase2/c0_v1_results'
from src.phase2.c0_views import load_results, style, PLOTS
style()
data = load_results(SUMMARY_DIR)
assert data['integrity']['status'] == 'passed'
print('Read-only aggregate artifacts loaded. Final integrity check passed.')

## Diagnostics and visualizations
All scores use ER+=1, ER−=0, and the unchanged probability link and 0.5 threshold. The 835 development patients supply five canonical folds; the 209 locked-test patients were excluded. Each training fold has 668 patients and validation fold 167. Primary differences are Q−E within a schedule. Standard deviations and plotted folds describe variation, not confidence intervals.

In [ ]:
display(data['primary_summary'].query("split == 'validation'").round(6))

### Primary Q−E contrasts

In [ ]:
figure = PLOTS['primary_contrasts'](data)
plt.show()

NQE has a positive ROC-AUC fold-mean contrast in all five folds, but log loss favors Q in three and E in two. Hybrid Q worsens both endpoints on average. Joint endpoints must be interpreted together; no favorable fold or seed is selected.

### Final VQC predictions

In [ ]:
figure = PLOTS['predictive_scores'](data)
plt.show()

NQE-Q reaches mean validation ROC-AUC 0.9300 and log loss 0.2958 nats. Historical B is contextual (0.9215, 0.2867 nats): discrimination improves slightly, probability loss worsens. Budgets and procedures differ.

### Learning curves

In [ ]:
figure = PLOTS['learning_curves'](data)
plt.show()

NQE curves for BCE begin after 50 geometric epochs; hybrids train jointly. The x-axis is within-stage epoch and is not equal elapsed compute. Online training BCE and end-of-epoch validation BCE are not same-checkpoint gap estimates. Final generalization below uses the same checkpoint.

### Same-checkpoint generalization

In [ ]:
figure = PLOTS['generalization'](data)
plt.show()

All four arms approach perfect training discrimination. Validation−training loss gaps are 0.253–0.308 nats; Q does not reliably reduce specialization. A smaller training loss is not sufficient evidence of improved generalization.

### Geometry transfer

In [ ]:
figure = PLOTS['geometry_transfer'](data)
plt.show()

All learned encoders improve state geometry relative to initialization. The strong NQE-Q training separation transfers incompletely: validation trace distances for NQE-Q and NQE-E are nearly the same despite different training distances. Trace distance and pair risk answer distinct questions.

### Frozen and classical readouts

In [ ]:
figure = PLOTS['readout_controls'](data)
plt.show()

All final encoders support competitive classical probes. LR-z and the fixed Euclidean overlap readout on NQE-Q outperform its VQC in mean ROC-AUC and log loss. This does not prove that VQC gradients were irrelevant to hybrid encoder learning. Frozen v₀ is an orientation-specific control, not a trained-classifier benchmark.

### State distance, observable response and prediction

In [ ]:
figure = PLOTS['accessibility'](data)
plt.show()

A high ratio of absolute raw class-mean contrast to trace distance coexists with a classical-readout gap. The ratio is not a complete measure of discriminative capacity or ROC-AUC. Neither ensemble separation nor a large mean response guarantees predictive generalization.

### Gradient stability

In [ ]:
figure = PLOTS['gradient_components'](data)
plt.show()

The fixed lambda produces auxiliary-to-BCE encoder-gradient norm ratios near one at the final checkpoint on diagnostic batches. Mean gradient cosines are mildly negative. These are observed local interactions, not proof that gradient conflict caused the predictive outcome.

### Computational cost

In [ ]:
figure = PLOTS['costs'](data)
plt.show()

The primary process took 216.875 seconds on CPU with one Torch thread and peak RSS 553,811,968 bytes. Pair losses use exact classical formulas. Stage timers are a partial decomposition; initial scoring, some frozen-response diagnostics, input loading and serialization are additional costs. No hardware shots or computational quantum advantage are claimed.

## Interpretation and limitations

The conditional result favoring RY fidelity within two-stage training is worth explaining, but its absence in the hybrid schedule and the performance of classical readouts weaken a general quantum-specific account. The evidence supports a gap between optimizing state-pair geometry and the predictions of the restricted classifier; it does not identify a new method or establish novelty.

These are reused development folds, with overlapping training sets and repeated patients across seeds. No significance tests, naive independent-seed intervals, retrospective hyperparameter selection or locked-test evaluation are performed. The future scientific question is whether a concrete, falsifiable mechanism can explain the geometry/readout/generalization mismatch beyond classical supervision and optimization; no C1 function is proposed here.

For exact per-fold/per-seed numbers, failures, provenance and reproduction commands, read `reports/phase2/c0_v1_results/report.md`. The source notebook deliberately contains no embedded outputs; an executed local copy is retained under the ignored run artifacts.